In [ ]:
## AI Use Statement

This assignment was completed with the assistance of **GitHub Copilot** (powered by Claude). The tool was used to:
- Assist with structuring the Jupyter Notebook and generating Python code for network analysis tasks.
- Help with debugging and refining NetworkX code.
- Generate Markdown documentation and assist with interpreting results.

All outputs were reviewed and validated by the student to ensure correctness and understanding.

In [ ]:
---
# Task 1: Static Network Construction

We construct a **weighted directed graph** from Dublin bike rental data. Each node represents a unique bike station, and each directed edge represents one or more trips between stations, weighted by trip count.

In [ ]:
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import numpy as np

,station_id,name,capacity,area
0,ST_0001,Smithfield North,42,Northside
1,ST_0002,Parnell Square North,45,Northside
2,ST_0003,Clonmel Street,40,Southside
3,ST_0004,Avondale Road,40,Northside
4,ST_0005,Mount Street Lower,38,Southside


In [ ]:
# Load the datasets
df_s = pd.read_csv('stations.csv')
df_r = pd.read_csv('rentals.csv')

print(f"Stations loaded: {len(df_s)}")
print(f"Rentals loaded:  {len(df_r)}")

,year,month,day,hour,start_station_id,end_station_id
0,2026,1,27,19,ST_0018,ST_0057
1,2026,1,26,16,ST_0028,ST_0041
2,2026,1,31,7,ST_0013,ST_0060
3,2026,1,28,10,ST_0064,ST_0095
4,2026,1,26,12,ST_0020,ST_0042


In [ ]:
df_s.head()

In [ ]:
df_r.head()

In [ ]:
### Building the Weighted Directed Network

We model bike trips as a **weighted directed graph**:
- **Nodes** represent stations, with attributes for station name, area, and capacity.
- **Edges** are directed from origin to destination station, with a **weight** equal to the number of trips on that route.
- **Self-loops** (trips where origin = destination) are excluded, as the specification states edges represent trips to a *different* destination.

In [ ]:
# Filter out self-loops
df_trips = df_r[df_r['start_station_id'] != df_r['end_station_id']].copy()
print(f"Valid trips (excluding self-loops): {len(df_trips)}")

# Aggregate trip counts per route to get edge weights
edge_weights = df_trips.groupby(['start_station_id', 'end_station_id']).size().reset_index(name='weight')

# Create weighted directed graph
G = nx.DiGraph()

# Add nodes with attributes
for _, row in df_s.iterrows():
    G.add_node(row['station_id'], name=row['name'], area=row['area'], capacity=row['capacity'])

# Add weighted edges
for _, row in edge_weights.iterrows():
    G.add_edge(row['start_station_id'], row['end_station_id'], weight=row['weight'])

print(f"\nNodes: {G.number_of_nodes()}")
print(f"Edges (unique routes): {G.number_of_edges()}")
print(f"Total trips represented: {sum(nx.get_edge_attributes(G, 'weight').values())}")

Nodes: 110
Edges: 1241
Density: 0.10350291909924937


In [ ]:
# Basic network statistics
print("=== Basic Network Statistics ===")
print(f"Network Density: {nx.density(G):.4f}")
print(f"Is strongly connected: {nx.is_strongly_connected(G)}")
print(f"Is weakly connected:   {nx.is_weakly_connected(G)}")
print(f"Strongly connected components: {nx.number_strongly_connected_components(G)}")
print(f"Weakly connected components:   {nx.number_weakly_connected_components(G)}")

Top 5 Accumulating Stations (Positive Flow):
   station_id  net_flow                               name
94    ST_0095        50  Princes Street / O'Connell Street
67    ST_0068        42          St. Stephen'S Green South
0     ST_0001        30                   Smithfield North
1     ST_0002        30               Parnell Square North
65    ST_0066        26                Merrion Square East

Top 5 Depleting Stations (Negative Flow):
   station_id  net_flow              name
8     ST_0009       -13  York Street East
86    ST_0087       -12   Parkgate Street
87    ST_0088       -11       Dame Street
93    ST_0094       -11  Charleville Road
11    ST_0012       -10   Portobello Road


In [ ]:
---
# Task 2: Static Network Characterisation

## 2a) Network Structure & Connectivity

We examine key structural properties of the network to understand the overall topology of Dublin's bike-sharing system.

Top Hubs by Betweenness Centrality:
   station_id    degree  betweenness  pagerank  \
65    ST_0066  0.605505     0.285891  0.022200   
38    ST_0039  0.284404     0.133990  0.008183   
67    ST_0068  0.715596     0.104559  0.034263   
94    ST_0095  0.752294     0.096488  0.050712   
12    ST_0013  0.137615     0.053196  0.003690   

                                 name  
65                Merrion Square East  
38                    Kilmainham Gaol  
67          St. Stephen'S Green South  
94  Princes Street / O'Connell Street  
12       St. James Hospital (Central)  


In [ ]:
# Degree distribution analysis
in_degrees = [d for n, d in G.in_degree()]
out_degrees = [d for n, d in G.out_degree()]

print("=== Degree Statistics ===")
print(f"Average in-degree:  {np.mean(in_degrees):.2f}")
print(f"Average out-degree: {np.mean(out_degrees):.2f}")
print(f"Max in-degree:  {max(in_degrees)}")
print(f"Max out-degree: {max(out_degrees)}")

# Reciprocity — fraction of edges that have a reverse edge
print(f"\nReciprocity: {nx.reciprocity(G):.4f}")

# Clustering coefficient (computed on undirected version)
G_undirected = G.to_undirected()
print(f"Average clustering coefficient: {nx.average_clustering(G_undirected):.4f}")
print(f"Transitivity: {nx.transitivity(G_undirected):.4f}")

# Diameter and average shortest path length
if nx.is_strongly_connected(G):
    print(f"\nDiameter: {nx.diameter(G)}")
    print(f"Average shortest path length: {nx.average_shortest_path_length(G):.4f}")
else:
    largest_scc = max(nx.strongly_connected_components(G), key=len)
    G_scc = G.subgraph(largest_scc)
    print(f"\nLargest strongly connected component: {len(largest_scc)} nodes")
    print(f"Diameter (largest SCC): {nx.diameter(G_scc)}")
    print(f"Avg shortest path length (largest SCC): {nx.average_shortest_path_length(G_scc):.4f}")

# Degree distribution plots
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(in_degrees, bins=20, alpha=0.7, color='steelblue', edgecolor='black')
axes[0].set_title('In-Degree Distribution')
axes[0].set_xlabel('In-Degree')
axes[0].set_ylabel('Frequency')

axes[1].hist(out_degrees, bins=20, alpha=0.7, color='coral', edgecolor='black')
axes[1].set_title('Out-Degree Distribution')
axes[1].set_xlabel('Out-Degree')
axes[1].set_ylabel('Frequency')

plt.tight_layout()
plt.show()

Subgraph created with 43 nodes and 453 edges.


In [ ]:
### Interpretation: Network Structure

- **Density**: The network density indicates what fraction of all possible station-to-station routes are actually used. A low density suggests riders tend to travel on specific routes rather than all possible pairings.
- **Connectivity**: Whether the graph is strongly or weakly connected tells us about the reachability of stations. Strong connectivity means every station can be reached from every other station by following directed edges — important for system usability.
- **Reciprocity**: High reciprocity indicates that routes tend to be bidirectional — if people travel from A to B, they also frequently travel from B to A (e.g., commuting patterns).
- **Clustering**: The clustering coefficient reveals whether stations that share connections also tend to be connected to each other, forming tightly-knit local clusters.
- **Degree Distribution**: The degree distributions show whether trip activity is spread evenly or concentrated at a few hub stations.

## 2b) Centrality Analysis

We apply multiple centrality measures to identify the most important hub stations in the network.

In [ ]:
# Calculate multiple centrality measures
degree_cent = nx.degree_centrality(G)
in_degree_cent = nx.in_degree_centrality(G)
out_degree_cent = nx.out_degree_centrality(G)
between_cent = nx.betweenness_centrality(G, weight='weight')
closeness_cent = nx.closeness_centrality(G)
pagerank_cent = nx.pagerank(G, weight='weight')

# Create centrality DataFrame
centrality_df = pd.DataFrame({
    'station_id': list(G.nodes()),
    'name': [G.nodes[n]['name'] for n in G.nodes()],
    'degree': [degree_cent[n] for n in G.nodes()],
    'in_degree': [in_degree_cent[n] for n in G.nodes()],
    'out_degree': [out_degree_cent[n] for n in G.nodes()],
    'betweenness': [between_cent[n] for n in G.nodes()],
    'closeness': [closeness_cent[n] for n in G.nodes()],
    'pagerank': [pagerank_cent[n] for n in G.nodes()]
})

print("=== Top 5 by Degree Centrality ===")
print(centrality_df.nlargest(5, 'degree')[['name', 'degree']].to_string(index=False))

print("\n=== Top 5 by Betweenness Centrality ===")
print(centrality_df.nlargest(5, 'betweenness')[['name', 'betweenness']].to_string(index=False))

print("\n=== Top 5 by Closeness Centrality ===")
print(centrality_df.nlargest(5, 'closeness')[['name', 'closeness']].to_string(index=False))

print("\n=== Top 5 by PageRank ===")
print(centrality_df.nlargest(5, 'pagerank')[['name', 'pagerank']].to_string(index=False))

In [ ]:
### Interpretation: Centrality Analysis

- **Degree Centrality** measures the fraction of other stations each station is directly connected to. High degree centrality indicates a station that is a general junction point, with routes to and from many other stations.
- **Betweenness Centrality** identifies stations that lie on the shortest paths between many other station pairs. These stations act as critical bridges in the network — if they were removed, many routes would become longer or severed.
- **Closeness Centrality** measures how close (in terms of path length) a station is to all other stations. High closeness means a station can be reached efficiently from anywhere in the network.
- **PageRank** accounts for both the quantity and quality of incoming connections. A station with high PageRank receives trips from other important stations, indicating it is a significant destination.

Stations that rank highly across multiple centrality measures are the **true hub stations** of the Dublin bike-sharing network — they are both well-connected and structurally important.

## 2c) Net Flow Analysis

We identify stations with significant positive net flow (more arrivals than departures — accumulating bikes) and negative net flow (more departures — depleting bikes).

In [ ]:
# Calculate net flow: arrivals minus departures
arrivals = df_trips['end_station_id'].value_counts()
departures = df_trips['start_station_id'].value_counts()

flow_df = pd.DataFrame({
    'station_id': df_s['station_id'],
    'name': df_s['name'],
    'area': df_s['area'],
    'arrivals': df_s['station_id'].map(arrivals).fillna(0).astype(int),
    'departures': df_s['station_id'].map(departures).fillna(0).astype(int)
})
flow_df['net_flow'] = flow_df['arrivals'] - flow_df['departures']

print("=== Top 5 Accumulating Stations (Positive Net Flow — more arrivals) ===")
print(flow_df.nlargest(5, 'net_flow')[['name', 'area', 'arrivals', 'departures', 'net_flow']].to_string(index=False))

print("\n=== Top 5 Depleting Stations (Negative Net Flow — more departures) ===")
print(flow_df.nsmallest(5, 'net_flow')[['name', 'area', 'arrivals', 'departures', 'net_flow']].to_string(index=False))

# Visualize net flow distribution
fig, ax = plt.subplots(figsize=(14, 5))
flow_sorted = flow_df.sort_values('net_flow')
colors = ['#d32f2f' if x < 0 else '#388e3c' for x in flow_sorted['net_flow']]
ax.bar(range(len(flow_sorted)), flow_sorted['net_flow'], color=colors, alpha=0.8)
ax.set_xlabel('Station (sorted by net flow)')
ax.set_ylabel('Net Flow (Arrivals − Departures)')
ax.set_title('Net Flow Distribution Across All Stations')
ax.axhline(y=0, color='black', linestyle='-', linewidth=0.5)
plt.tight_layout()
plt.show()

### Interpretation: Net Flow

Stations with **significant positive net flow** (green bars) receive more bikes than they send out — they are *accumulating* bikes over the observation period. These tend to be located in areas that people travel **to**, such as commercial centres or workplaces.

Stations with **significant negative net flow** (red bars) send out more bikes than they receive — they are *depleting*. These are typically in residential areas where people **start** their journeys.

This imbalance has important operational implications: the bike-sharing operator needs to **physically redistribute bikes** from accumulating stations back to depleting stations to maintain service availability across the network.

---
# Task 3: Static Network Visualisation

## 3a) Northside Subgraph Extraction

We extract a subgraph containing only stations in the **'Northside'** area and export it in GEXF format for visualisation in Gephi.

In [ ]:
# Extract Northside subgraph
northside_nodes = [n for n, attr in G.nodes(data=True) if attr.get('area') == 'Northside']
S = G.subgraph(northside_nodes).copy()

print(f"Northside subgraph: {S.number_of_nodes()} nodes, {S.number_of_edges()} edges")
print(f"Density: {nx.density(S):.4f}")

# Export to GEXF for Gephi
nx.write_gexf(S, "northside_subgraph.gexf")
print("\nExported 'northside_subgraph.gexf' successfully.")

## 3b) Gephi Visualisation

The GEXF file was loaded into Gephi and visualised using the following approach:

1. **Layout**: The *ForceAtlas 2* layout algorithm was applied to spatially arrange nodes based on connectivity strength, pulling strongly connected stations closer together and revealing natural clusters.
2. **Node Size**: Nodes were sized proportionally to their **weighted degree** (total trip volume) to visually highlight the most active stations.
3. **Edge Thickness**: Edges were scaled by their **weight** (trip count) so that the most popular routes stand out.
4. **Node Colour**: Nodes were coloured by **modularity class** (computed using Gephi's modularity algorithm) to identify communities of stations that interact more with each other than with the rest of the network.

The resulting visualisations are exported as PNG files and included in the submission package.

---
# Task 4: Dynamic Network Analysis

## 4a) Daily Time Window Networks

We construct separate weighted directed networks for each day in the dataset (7 days: Mon 26 Jan – Sun 1 Feb 2026) to analyse how the bike-sharing network evolves over time.

In [ ]:
# Add date column and filter self-loops
df_r['date'] = pd.to_datetime(df_r[['year', 'month', 'day']])
df_r_clean = df_r[df_r['start_station_id'] != df_r['end_station_id']].copy()

# Get unique dates sorted
dates = sorted(df_r_clean['date'].unique())
print(f"Number of days in dataset: {len(dates)}")

# Construct daily networks
daily_graphs = {}
for date in dates:
    day_data = df_r_clean[df_r_clean['date'] == date]
    day_edges = day_data.groupby(['start_station_id', 'end_station_id']).size().reset_index(name='weight')
    
    G_day = nx.DiGraph()
    # Add all station nodes (so isolated stations are visible)
    for _, row in df_s.iterrows():
        G_day.add_node(row['station_id'], name=row['name'], area=row['area'], capacity=row['capacity'])
    # Add weighted edges for this day
    for _, row in day_edges.iterrows():
        G_day.add_edge(row['start_station_id'], row['end_station_id'], weight=row['weight'])
    
    day_label = pd.Timestamp(date).strftime('%a %d/%m')
    daily_graphs[day_label] = G_day
    active_nodes = sum(1 for n in G_day.nodes() if G_day.degree(n) > 0)
    print(f"{day_label}: {G_day.number_of_edges()} edges, {active_nodes} active stations, "
          f"{day_data.shape[0]} trips")

## 4b) Global Structure Metrics Over Time

We compute and compare global network metrics for each daily network to identify temporal patterns in how the bike-sharing system is used.

In [ ]:
# Compute global metrics for each daily network
metrics = []
for day_label, G_day in daily_graphs.items():
    # Focus on active subgraph (nodes with at least one edge)
    active_nodes = [n for n in G_day.nodes() if G_day.degree(n) > 0]
    G_active = G_day.subgraph(active_nodes)
    
    total_trips = sum(nx.get_edge_attributes(G_day, 'weight').values())
    
    m = {
        'Day': day_label,
        'Active Stations': len(active_nodes),
        'Unique Routes': G_day.number_of_edges(),
        'Total Trips': total_trips,
        'Density': nx.density(G_active) if len(active_nodes) > 1 else 0,
        'Reciprocity': nx.reciprocity(G_active) if G_active.number_of_edges() > 0 else 0,
        'Avg Clustering': nx.average_clustering(G_active.to_undirected()) if len(active_nodes) > 0 else 0,
    }
    
    # Diameter from largest strongly connected component
    if len(active_nodes) > 1:
        if nx.is_strongly_connected(G_active):
            m['Diameter'] = nx.diameter(G_active)
            m['Avg Path Length'] = round(nx.average_shortest_path_length(G_active), 2)
        else:
            largest_scc = max(nx.strongly_connected_components(G_active), key=len)
            G_scc = G_active.subgraph(largest_scc)
            m['Diameter'] = nx.diameter(G_scc)
            m['Avg Path Length'] = round(nx.average_shortest_path_length(G_scc), 2)
            m['SCC Size'] = len(largest_scc)
    
    metrics.append(m)

metrics_df = pd.DataFrame(metrics)
print(metrics_df.to_string(index=False))

In [ ]:
# Visualise key metrics over time
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

metrics_to_plot = [
    ('Total Trips', 'Total Trips per Day', 'steelblue'),
    ('Unique Routes', 'Unique Routes per Day', 'coral'),
    ('Active Stations', 'Active Stations per Day', 'green'),
    ('Density', 'Network Density per Day', 'purple'),
    ('Reciprocity', 'Reciprocity per Day', 'orange'),
    ('Avg Clustering', 'Avg Clustering per Day', 'teal'),
]

for ax, (col, title, color) in zip(axes.flatten(), metrics_to_plot):
    ax.plot(metrics_df['Day'], metrics_df[col], marker='o', color=color, linewidth=2, markersize=8)
    ax.set_title(title, fontsize=12)
    ax.set_xlabel('Day')
    ax.set_ylabel(col)
    ax.tick_params(axis='x', rotation=45)
    ax.grid(alpha=0.3)

plt.suptitle('Global Network Metrics Across Days', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

### Interpretation: Temporal Patterns

By examining global network metrics across the 7-day period (Mon 26 Jan – Sun 1 Feb 2026), several patterns emerge:

- **Weekday vs Weekend**: Weekdays typically show higher total trip volumes and more unique routes, reflecting commuting patterns. Weekend days may show different spatial distributions as trips become more leisure-oriented.
- **Network Density & Route Diversity**: Fluctuations in density and unique route count reflect changing travel patterns. Higher density days indicate more distributed travel across the network, while lower density suggests trips are concentrated on fewer routes.
- **Reciprocity Over Time**: Changes in reciprocity across days can indicate different usage types — commuting trips tend to have high reciprocity (same route there and back), while recreational rides may show lower reciprocity.
- **Clustering Variation**: Day-to-day changes in clustering reveal whether local station neighbourhoods maintain consistent connectivity or if patterns shift with the day of the week.

## 4c) Station Capacity Recommendations

We identify stations where capacity should be increased by examining **peak daily activity relative to station capacity** and **persistent net flow imbalances** over time.

In [ ]:
# Calculate daily activity per station across all days
station_daily_activity = []
for day_label, G_day in daily_graphs.items():
    for node in G_day.nodes():
        w_in = sum(d['weight'] for _, _, d in G_day.in_edges(node, data=True))
        w_out = sum(d['weight'] for _, _, d in G_day.out_edges(node, data=True))
        capacity = G_day.nodes[node].get('capacity', 0)
        
        station_daily_activity.append({
            'Day': day_label,
            'station_id': node,
            'name': G_day.nodes[node].get('name', ''),
            'area': G_day.nodes[node].get('area', ''),
            'capacity': capacity,
            'arrivals': w_in,
            'departures': w_out,
            'total_activity': w_in + w_out,
            'net_flow': w_in - w_out
        })

activity_df = pd.DataFrame(station_daily_activity)

# Aggregate station metrics across all days
station_summary = activity_df.groupby(['station_id', 'name', 'area', 'capacity']).agg(
    total_trips=('total_activity', 'sum'),
    avg_daily_trips=('total_activity', 'mean'),
    max_daily_trips=('total_activity', 'max'),
    avg_net_flow=('net_flow', 'mean'),
    max_abs_net_flow=('net_flow', lambda x: max(abs(x)) if len(x) > 0 else 0),
    days_active=('total_activity', lambda x: sum(x > 0))
).reset_index()

# Activity-to-capacity ratio: higher means the station may be under-provisioned
station_summary['activity_ratio'] = station_summary['max_daily_trips'] / station_summary['capacity']

# Rank stations for capacity increase
print("=== Top 10 Stations Where Capacity Should Be Increased ===")
print("(Ranked by peak daily activity relative to station capacity)\n")
capacity_candidates = station_summary.nlargest(10, 'activity_ratio')
print(capacity_candidates[['name', 'area', 'capacity', 'avg_daily_trips', 
                            'max_daily_trips', 'activity_ratio', 'avg_net_flow'
                           ]].round(2).to_string(index=False))

In [ ]:
# Visualize capacity analysis
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Scatter: Station Capacity vs Peak Daily Activity
axes[0].scatter(station_summary['capacity'], station_summary['max_daily_trips'], 
                alpha=0.6, color='steelblue', edgecolors='black', s=60)
axes[0].set_xlabel('Station Capacity (docks)', fontsize=11)
axes[0].set_ylabel('Peak Daily Trips', fontsize=11)
axes[0].set_title('Station Capacity vs Peak Daily Activity', fontsize=12)
max_val = max(station_summary['capacity'].max(), station_summary['max_daily_trips'].max())
axes[0].plot([0, max_val], [0, max_val], 'r--', alpha=0.5, label='1:1 reference line')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Horizontal bar: Top 10 stations by activity ratio
top10 = capacity_candidates.sort_values('activity_ratio', ascending=True)
axes[1].barh(top10['name'], top10['activity_ratio'], color='coral', edgecolor='black')
axes[1].set_xlabel('Activity Ratio (Peak Trips / Capacity)', fontsize=11)
axes[1].set_title('Top 10 Stations: Peak Activity / Capacity Ratio', fontsize=12)
axes[1].grid(alpha=0.3, axis='x')

plt.tight_layout()
plt.show()

### Interpretation: Capacity Recommendations

Stations where capacity should be increased are identified by the **activity-to-capacity ratio** — peak daily trip volume relative to the station's dock capacity:

- **High activity ratio** stations experience demand that approaches or exceeds their physical capacity, leading to situations where riders cannot find available bikes (at depleting stations) or cannot return bikes (at full accumulating stations).
- **Persistent net flow imbalance** compounds the problem: stations that consistently accumulate bikes need more empty docks, while consistently depleting stations need more bikes stocked.

**Recommendations:**
- The top-ranked stations above should have their dock capacity increased to handle peak demand without service disruption.
- Stations with consistently **high positive net flow** need more empty docks to accept incoming bikes.
- Stations with consistently **high negative net flow** need more bikes available to meet departure demand.
- The operator should prioritise capacity upgrades at stations that appear in both the high-activity and high-imbalance lists, as these face dual pressure from both volume and directional flow.
- Additionally, **time-of-day redistribution** should target these stations during peak hours to prevent service interruption.